# VALORACIÓN

Descuento de los flujos proyectados al costo de capital para obtener el valor de la operación, y de ahí el patrimonio.

$$
\text{Valor de la operación}
=
VP(\text{flujos explícitos})
+
VP(\text{valor terminal})
$$

$$
\text{Valor del patrimonio}
=
\text{Valor de la operación}
+
\text{efectivo}
-
\text{deuda}
$$


El crecimiento no se supone: se deriva de cuánto reinvierte la empresa y a qué retorno.

$$
g = \text{tasa de reinversión} \times ROC
$$



In [18]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt

# mdulo 1
ingresos_base = 38082.2
ebit_base = 2456.3
tasa_mx = 0.30
nopat = ebit_base * (1 - tasa_mx)

# modulo 2
deuda = 15935.7
efectivo = 1383.0
acciones = 555980425
precio_mercado = 11.89
rf_mxn = 0.0596

# modulo 3
roc_actual = 0.0597
roc_marginal = 0.0835

# modulo 4
wacc_actual = 0.1234
wacc_optimo = 0.1090

# supuestos de proyeccion
años = 5
g_alta = 0.030
g_estable = 0.030

print(f"NOPAT del año base : {nopat:,.1f}")
print(f"g estable {g_estable:.1%} contra techo de la Rf en pesos {rf_mxn:.2%}")

NOPAT del año base : 1,719.4
g estable 3.0% contra techo de la Rf en pesos 5.96%


In [19]:
trade_off = pd.DataFrame({"g": [0.00, 0.01, 0.02, 0.03, 0.05, 0.07]})
trade_off["reinversion_pct"] = trade_off["g"] / roc_actual
trade_off["reinversion"] = trade_off["reinversion_pct"] * nopat
trade_off["fcff"] = nopat - trade_off["reinversion"]

trade_off.round(3)

,g,reinversion_pct,reinversion,fcff
0,0.00,0.000,0.000,1719.410
1,0.01,0.168,288.008,1431.402
2,0.02,0.335,576.017,1143.393
3,0.03,0.503,864.025,855.385
4,0.05,0.838,1440.042,279.368
5,0.07,1.173,2016.059,-296.649


Con el ROC actual, crecer 7% (que proyecta Fitch en su caso base)dejaría el flujo libre negativo.  supone un ROC mayor, coherente con su proyección de margen EBITDAR subiendo de 15% a
16%.

## ROC promedio y ROC marginal

La fórmula del crecimiento debe usar el retorno del próximo peso invertido, no el promedio de todo el capital acumulado.

| | Capital | ROC |
|---|---|---|
| Promedio, capital total | 28,813.6 | 5.97% |
| Marginal, capital operativo | 20,596.4 | 8.35% |

La diferencia son 8,217 millones de crédito mercantil e intangibles: lo que Traxión pagó de más al comprar empresas. Ese costo ya está hundido y no se
repite. S

**Se adopta una convergencia lineal del ROC de 5.97% a 8.35% en cinco años.** El mecanismo no depende de creer nada sobre la administración: si el capital crece con inversión nueva y el crédito mercantil existente se mantiene fijo en 8,217, su peso en la base cae y el ROC promedio se acerca al marginal por dilución. Lo respalda lo anunciado en el 2T26, recorte de
capex, salida del 25% de la flota de carga, migración a asset-light, etc. que es lo contrario de seguir adquiriendo.

La convergencia exige reinversión: si el capital no crece, el crédito mercantil sigue pesando 28.5% indefinidamente y el ROC no mejora.

## Crecimiento de la fase alta: 3.0%

**Descartado el 7% de Fitch.** 

**Descartado el 3.2% del Módulo 1**

**Se adopta 3.0% nominal.** México crece 1.1-1.8% real según la encuesta de Banxico, con inflación esperada de 3.75%. Un crecimiento nominal de 3.0% implica volumen levemente negativo, consistente con una empresa que anunció salir del 25% de su flota de carga y cuyos kilómetros recorridos cayeron 5-6% en el semestre.

se supone que el crecimiento de logística compensa la contracción de carga, que es lo que viene ocurriendo.

In [20]:
#proyeccion
proy = pd.DataFrame({"año": range(1, años + 1)})
proy["roc"] = [roc_actual + (roc_marginal - roc_actual) * (t - 1) / (años - 1)
               for t in proy["año"]]
proy["nopat"] = nopat * (1 + g_alta) ** proy["año"]
proy["tasa_reinversion"] = g_alta / proy["roc"]
proy["reinversion"] = proy["nopat"] * proy["tasa_reinversion"]
proy["fcff"] = proy["nopat"] - proy["reinversion"]

proy.round(3)

,año,roc,nopat,tasa_reinversion,reinversion,fcff
0,1,0.060,1770.992,0.503,889.946,881.046
1,2,0.066,1824.122,0.457,833.567,990.555
2,3,0.072,1878.846,0.419,787.226,1091.620
3,4,0.078,1935.211,0.387,748.631,1186.580
4,5,0.084,1993.267,0.359,716.144,1277.123


## Estado estable

El valor terminal no se calcula con los supuestos de hoy. Una empresa madura es otra empresa: menos riesgosa, con más capacidad de deuda, con el retorno
sobre el capital erosionado hacia el costo de capital, y reinvirtiendo solo lo necesario para sostener su crecimiento.

La metodología es explícita en que todos esos supuestos cambian juntos. El error típico es mezclarlos: crecimiento de empresa madura con el beta y el ROC de empresa en crecimiento.

**Beta estable: 1.0.** Converge al del mercado, como corresponde a una empresa que dejó de ser más o menos cíclica que el promedio. Verificación:
con beta apalancado de 1.0 y el D/E de la estructura óptima (53.8%), el beta desapalancado implícito es 0.726, cercano al 0.793 estimado con las comparables. Los supuestos de madurez son coherentes con el negocio.

**Estructura estable: 35% de deuda**, el óptimo del Módulo 4. Una empresamadura tiene más capacidad de endeudamiento y no hay razón para suponer quemantiene indefinidamente una estructura no óptima que le cuesta.

**WACC estable: 10.49%.** Resulta de los dos anteriores más el costo de deuda del escalón óptimo, con rating Baa2/BBB y spread de 1.11%.

**ROC estable = WACC estable = 10.49%.** La metodología de Damodaran admite que el ROC supere al costo de capital a largo plazo, pero solo por poco, y como máximo converge a igualarlo. Traxión no tiene exceso perpetuo: opera en un sector fragmentado, dominado por operadores familiares que presionan precios, y la mitad de su negocio es brokerage con márgenes de
2-3% por diseño.

**Crecimiento estable: 3.0%**, con techo en la tasa libre de riesgo en pesos de 5.96%. Una empresa no puede crecer a perpetuidad más rápido que la
economía en la que opera.

**Reinversión estable: g / ROC = 3.0% / 10.49% = 28.6%**

El descuento del valor terminal a presente usa el WACC del período de proyección, no el estable: el valor terminal vive en el año 5 y hay que traerlo por los cinco años de la fase alta.

In [21]:
from src.costo_capital import convertir_tasa

beta_estable = 1.0
ratio_deuda_optimo = 0.35
spread_optimo = 0.0111
rf_usd = 0.0443
spread_mexico = 0.0152
erp_total = 0.0682
infl_mxn, infl_usd = 0.0375, 0.0225

ke_estable = convertir_tasa(rf_usd + beta_estable * erp_total, infl_mxn, infl_usd)
kd_estable = convertir_tasa(rf_usd + spread_mexico + spread_optimo,
                            infl_mxn, infl_usd) * (1 - tasa_mx)
wacc_estable = ke_estable * (1 - ratio_deuda_optimo) + kd_estable * ratio_deuda_optimo

roc_estable = wacc_estable

print(f"Ke estable              : {ke_estable:.2%}")
print(f"kd estable despues de t : {kd_estable:.2%}")
print(f"WACC estable            : {wacc_estable:.2%}")
print(f"ROC estable             : {roc_estable:.2%}")
print(f"Reinversion estable     : {g_estable/roc_estable:.1%}")
print()
de_implicito = ratio_deuda_optimo / (1 - ratio_deuda_optimo)
print(f"Beta desapalancado implicito: {beta_estable/(1+(1-tasa_mx)*de_implicito):.3f}")
print(f"Beta desapalancado estimado : {0.793:.3f}")

Ke estable              : 12.88%
kd estable despues de t : 6.04%
WACC estable            : 10.49%
ROC estable             : 10.49%
Reinversion estable     : 28.6%

Beta desapalancado implicito: 0.726
Beta desapalancado estimado : 0.793


In [22]:
#DCF
def valorar(wacc_proyeccion, wacc_estable, roc_estable, proyeccion,
            g_estable, deuda, efectivo, acciones):
    """Descuenta los flujos proyectados y el valor terminal, y arma el puente al
    patrimonio. El valor terminal usa el WACC de madurez; el descuento a presente
    usa el WACC del periodo de proyeccion"""
    nopat_terminal = proyeccion["nopat"].iloc[-1] * (1 + g_estable)
    reinversion_estable = g_estable / roc_estable
    fcff_terminal = nopat_terminal * (1 - reinversion_estable)
    valor_terminal = fcff_terminal / (wacc_estable - g_estable)

    factores = 1 / (1 + wacc_proyeccion) ** proyeccion["año"]
    vp_explicito = (proyeccion["fcff"] * factores).sum()
    vp_terminal = valor_terminal * factores.iloc[-1]
    valor_operacion = vp_explicito + vp_terminal
    valor_patrimonio = valor_operacion + efectivo - deuda

    return {
        "wacc_proyeccion": wacc_proyeccion,
        "valor_terminal": valor_terminal,
        "vp_explicito": vp_explicito,
        "vp_terminal": vp_terminal,
        "peso_terminal": vp_terminal / valor_operacion,
        "valor_operacion": valor_operacion,
        "valor_patrimonio": valor_patrimonio,
        "valor_por_accion": valor_patrimonio * 1e6 / acciones,
    }


escenarios = pd.DataFrame([
    valorar(w, wacc_estable, roc_estable, proy, g_estable, deuda, efectivo, acciones)
    for w in [wacc_actual, wacc_optimo]
], index=["estructura actual", "estructura optima"])

escenarios.round(2)

,wacc_proyeccion,valor_terminal,vp_explicito,vp_terminal,peso_terminal,valor_operacion,valor_patrimonio,valor_por_accion
estructura actual,0.12,19575.74,3797.9,10940.72,0.74,14738.62,185.92,0.33
estructura optima,0.11,19575.74,3946.0,11669.72,0.75,15615.72,1063.02,1.91


### Resultado

| | Valor de la operación | Patrimonio | Por acción |
|---|---|---|---|
| Estructura actual, WACC 12.34% | 14,738.6 | 185.9 | 0.33 |
| Estructura óptima, WACC 10.90% | 15,615.7 | 1,063.0 | 1.91 |

Desapalancar hasta el óptimo vale 877 millones, unos 1.58 pesos por acción. Es el costo del sobreapalancamiento.

El valor terminal pesa entre 74% y 75% del total. Está dentro del rango de 60-80% que la metodología considera normal para una empresa madura con proyección corta. 

**La brecha con el mercado es grande.** El precio de 11.89 implica un patrimonio de 6,610.6 y un valor de la firma de 21,163, contra los 15,616 que arroja el modelo con la estructura óptima. El mercado valora la operación 36% por encima.

Esa diferencia se debe principalmente an los supuestos operativos de la fase de proyección: margen base de 6.45% y crecimiento de 3.0%, ambos más conservadores que el caso base de Fitch, que proyecta margen EBITDAR mejorando a 16% y crecimiento de 7%.

## Valoración inversa

El modelo arroja 1.91 por acción con la estructura óptima, contra un precio de mercado de 11.89. La brecha es grande.

La valoración inversa responde qué habría que suponer para llegar al precio observado. No se trata de ajustar el modelo al mercado, sino de identificar con precisión qué está descontando el mercado que este análisis no descuenta.

In [23]:
def valor_patrimonio(g=g_alta, wacc=wacc_optimo, margen=None,
                     roc_fin=roc_marginal, w_est=wacc_estable, roc_est=roc_estable):
    """Recalcula el patrimonio variando un supuesto a la vez"""
    nop = ingresos_base * margen * (1 - tasa_mx) if margen else nopat

    vp = 0
    for a in range(1, años + 1):
        roc = roc_actual + (roc_fin - roc_actual) * (a - 1) / (años - 1)
        n = nop * (1 + g) ** a
        vp += n * (1 - g / roc) / (1 + wacc) ** a
        if a == años:
            n_final = n

    vt = n_final * (1 + g) * (1 - g / roc_est) / (w_est - g)
    return vp + vt / (1 + wacc) ** años + efectivo - deuda


sens_g = pd.DataFrame({"g": [0.00, 0.02, 0.03, 0.04, 0.05]})
sens_g["patrimonio"] = [valor_patrimonio(g=x) for x in sens_g["g"]]

sens_w = pd.DataFrame({"wacc": [0.080, 0.090, 0.095, 0.100, 0.109]})
sens_w["patrimonio"] = [valor_patrimonio(wacc=x, w_est=x, roc_est=x)
                        for x in sens_w["wacc"]]

sens_m = pd.DataFrame({"margen": [0.0493, 0.0645, 0.0766, 0.0874]})
sens_m["patrimonio"] = [valor_patrimonio(margen=x) for x in sens_m["margen"]]

print(sens_g.to_string(index=False, formatters={
    "g": "{:.1%}".format, "patrimonio": "{:,.1f}".format}))
print()
print(sens_w.to_string(index=False, formatters={
    "wacc": "{:.2%}".format, "patrimonio": "{:,.1f}".format}))
print()
print(sens_m.to_string(index=False, formatters={
    "margen": "{:.2%}".format, "patrimonio": "{:,.1f}".format}))

   g patrimonio
0.0%    1,591.3
2.0%    1,254.4
3.0%    1,063.0
4.0%      855.8
5.0%      632.2

  wacc patrimonio
 8.00%    7,186.3
 9.00%    4,429.0
 9.50%    3,274.1
10.00%    2,238.4
10.90%      621.7

margen patrimonio
 4.93%   -2,617.0
 6.45%    1,063.0
 7.66%    3,992.5
 8.74%    6,607.2


### El crecimiento destruye valor

**El valor cae cuando el crecimiento sube.** No existe ninguna tasa de crecimiento que lleve el valor al precio de mercado.

La razón es la que viene documentándose desde el Módulo 2: durante la fase
de proyección el ROC va de 5.97% a 8.35%, siempre por debajo del WACC de
10.90%. 

### Qué habría que suponer para llegar al precio de mercado

| Supuesto | Valor necesario | Base adoptada | Evaluación |
|---|---|---|---|
| Margen operativo | 8.74% | 6.45% | Supera el techo de recuperación total de los tres segmentos (7.66%) |
| WACC | 8.0% | 10.90% | Casi tres puntos por debajo |
| Crecimiento | no existe | 3.0% | Ninguna tasa alcanza el precio |

Ninguno por sí solo es plausible. Pero una combinación sí:

$$
7.66\% + 9.5\% = 11.90\ \text{por acción}
$$

El mercado estaría descontando dos cosas a la vez:

**Que los tres segmentos recuperan sus márgenes de hace un año.** Es el escenario de recuperación total del Módulo 1

**Y un costo de capital de 9.5%**, unos 140 puntos básicos menor.

Esa segunda diferencia tiene explicación conocida. El WACC de este modelo usa un rating sintético de B3/B-, derivado de la tabla de empresas pequeñas aplicada a una cobertura de 1.64x. Fitch califica a Traxión en A+(mex). 

La brecha identificada en el Módulo 2 entre rating sintético y real reaparece aquí, PERO EN VALOR.

## Conclusión del módulo

| | Valor de la operación | Patrimonio | Por acción |
|---|---|---|---|
| Estructura actual | 14,738.6 | 185.9 | 0.33 |
| Estructura óptima | 15,615.7 | 1,063.0 | 1.91 |
| Precio de mercado | 21,163 | 6,610.6 | 11.89 |

**El valor depende críticamente del margen operativo.** 

**El sobreapalancamiento cuesta 877 millones**, 1.58 pesos por acción. Es la
diferencia entre operar con 70.7% de deuda y con el óptimo de 35%.

**El crecimiento destruye valor** mientras el ROC esté por debajo del costo de capital. 

**La brecha con el mercado se explica por dos supuestos.** El precio de 11.89 requiere margen de 7.66% (el techo de recuperación total) y un costo de capital de 9.5%. Este análisis adoptó 6.45% y 10.90%.

Ninguno de los dos supuestos del mercado es descabellado. El margen de 7.66% es uno de los cuatro escenarios que se evaluaron en el Módulo 1 y se descartó por ser muy optimista; el WACC de 9.5% sería consistente con la calificación A+(mex) de Fitch en vez del B3/B− del rating sintético.

La conclusión es que el mercado descuenta el escenario optimista sobre el que este análisis no aposto.